# Double Zernike Analysis (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-12  
**Last Modified:** 2026-04-12  
**Status:** Draft  
**Keywords:** Double Zernike, AOS, intrinsics, correlations, LUT, rotator, elevation, thermal  

## Description

Aggregate Double Zernike (DZ) fit results across all pipeline chunks (OCS or CCS
coordinate system) and run cross-coefficient and operational analyses.

Key functionality:
1. Load and concatenate all `*_fits.parquet` (OCS) or `*_ccs_fits.parquet` (CCS)
   from the pipeline `output/` tree.
2. Reproduce DZ-vs-thermal scatter plots from `intrinsics_thermal_correlations`.
3. Full pairwise correlation analysis over all DZ coefficients; pull out the
   10 largest off-diagonal correlations and make scatter plots.
4. LUT exploration: DZ terms vs rotator angle (elevation ∈ 70 ± 3 deg) and
   DZ terms vs elevation (rotator angle ∈ 0 ± 3 deg).

**Output:** PDF in `output/dz_analysis_{coord_sys}.pdf`  
**Based on:** intrinsics_thermal_correlations.ipynb, run_pipeline outputs

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-12 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Thermal Correlations](#thermal)
6. [Full DZ Correlation Analysis](#corr)
7. [LUT: DZ vs Rotator and Elevation](#lut)
8. [Cylindrical (Radial) Pupil Zernike Combinations](#radial)
9. [Output](#output)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Coordinate system for fit tables: 'OCS' or 'CCS'
coord_sys = 'OCS'

# Where to look for fit parquet files (relative to this notebook)
output_root = 'output'

# Glob patterns: OCS uses *_fits.parquet, CCS uses *_ccs_fits.parquet
fit_glob = {
    'OCS': '*[!_ccs]_fits.parquet',
    'CCS': '*_ccs_fits.parquet',
}[coord_sys]

# DZ fit prefix to use for analysis (z1toz3 or z1toz6)
dz_prefix = 'z1toz6'

# Quality cuts
max_coeff_um = 2.0  # reject visits with any DZ coefficient > this (µm)

# Thermal variables (only used if present in the merged tables)
thermal_vars = [
    'cam_air_temp', 'm2_air_temp', 'm1m3_air_temp', 'outside_temp',
    'm2_delta_t', 'cam_m1m3_delta_t', 'dome_delta_t',
    'x_gradient', 'y_gradient', 'z_gradient', 'radial_gradient',
    'tma_truss_temp_pxpy', 'tma_truss_temp_mxmy',
]

# DZ terms to plot vs thermal (k = focal Noll, j = pupil Noll).
# Include all focal k for pupil j = 4 (defocus) plus astig/trefoil combos.
dz_terms_for_thermal = [
    (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4),  # all k, pupil defocus
    (5, 5), (6, 6),                                    # astig-astig
    (5, 10), (6, 9),                                   # astig-trefoil
]

# Top-N largest correlations to scatter-plot
top_n_corr = 50

# Expected astigmatism-symmetry correlations in the (k, j) basis.
# Each tuple: ((k_a, j_a), (k_b, j_b)) — one scatter per tuple.
expected_astig_pairs = [
    ((5, 5), (6, 6)),
    ((6, 5), (5, 6)),
    ((1, 5), (1, 6)),
    ((4, 5), (4, 6)),
    ((2, 5), (3, 6)),
    ((3, 5), (2, 6)),
]

# LUT slice tolerances (all in degrees)
elev_center_for_rotator_scan = 70.0
elev_tol = 3.0
rot_center_for_elev_scan = 0.0
rot_tol = 3.0

# Output PDF
output_pdf = f'{output_root}/dz_analysis_{coord_sys}.pdf'

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

<a id='functions'></a>
## Helper Functions

In [ ]:
def find_fit_files(output_root, coord_sys):
    """Locate all DZ fit parquet files for the requested coord system.

    OCS:  files ending in '_fits.parquet' but NOT '_ccs_fits.parquet'.
    CCS:  files ending in '_ccs_fits.parquet'.
    """
    root = Path(output_root)
    all_fits = sorted(root.glob('*_fits.parquet'))
    if coord_sys == 'CCS':
        return [p for p in all_fits if p.name.endswith('_ccs_fits.parquet')]
    return [p for p in all_fits if not p.name.endswith('_ccs_fits.parquet')]


def load_all_fits(files, source_label='source'):
    """Load and concatenate fit parquet tables, tagging each row with its source."""
    frames = []
    for p in files:
        df = pd.read_parquet(p)
        df[source_label] = p.stem
        frames.append(df)
        print(f"  {p.name}: {len(df)} rows, {len(df.columns)} cols")
    if not frames:
        raise FileNotFoundError('No fit parquet files found')
    combined = pd.concat(frames, ignore_index=True, sort=False)
    return combined


def dz_coeff_columns(df, prefix):
    """Return all DZ coefficient columns for a given prefix.

    Columns are named '{prefix}_z{j}_c{k}' where j = pupil Noll, k = focal Noll.
    Excludes '_err', '_scale', '_bad_fit'.
    """
    pattern = re.compile(rf'^{re.escape(prefix)}_z\d+_c\d+$')
    return [c for c in df.columns if pattern.match(c)]


def parse_jk(col, prefix):
    """Parse '(j, k)' from a column like 'z1toz6_z9_c5'."""
    m = re.match(rf'^{re.escape(prefix)}_z(\d+)_c(\d+)$', col)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))


def short_label(col, prefix):
    j, k = parse_jk(col, prefix)
    return f'(k={k}, j={j})'


def quality_cut(df, prefix, max_coeff_um):
    """Apply standard quality cuts: drop bad_fit and outliers."""
    n0 = len(df)
    bad_col = f'{prefix}_bad_fit'
    if bad_col in df.columns:
        df = df[~df[bad_col].astype(bool)].copy()
    elif 'bad_fit' in df.columns:
        df = df[~df['bad_fit'].astype(bool)].copy()
    cols = dz_coeff_columns(df, prefix)
    out_mask = df[cols].abs().gt(max_coeff_um).any(axis=1)
    df = df[~out_mask].copy()
    print(f'Quality cut: {len(df)}/{n0} rows kept '
          f'(prefix={prefix}, |c| < {max_coeff_um} µm)')
    return df

<a id='data'></a>
## Data Access

In [ ]:
fit_files = find_fit_files(output_root, coord_sys)
print(f'Found {len(fit_files)} {coord_sys} fit files in {output_root}/')
for p in fit_files:
    print(f'  {p}')

df_all = load_all_fits(fit_files, source_label='chunk')
print(f'\nCombined: {len(df_all)} rows x {len(df_all.columns)} columns')
print(f'day_obs range: {df_all["day_obs"].min()} - {df_all["day_obs"].max()}')

In [ ]:
df = quality_cut(df_all, dz_prefix, max_coeff_um)
dz_cols = dz_coeff_columns(df, dz_prefix)
print(f'Number of DZ coefficient columns: {len(dz_cols)}')

# Unit normalization:
#   * alt is stored in RADIANS (from Butler meta) -> add alt_deg in degrees
#   * rotator_angle is already in DEGREES
if 'alt' in df.columns:
    df['alt_deg'] = np.rad2deg(df['alt'].astype(float))
    print(f'alt range (rad): {df["alt"].min():.3f} .. {df["alt"].max():.3f}')
    print(f'alt range (deg): {df["alt_deg"].min():.2f} .. {df["alt_deg"].max():.2f}')
if 'rotator_angle' in df.columns:
    print(f'rotator_angle range (deg): '
          f'{df["rotator_angle"].min():.2f} .. {df["rotator_angle"].max():.2f}')

# Sanity check on key metadata columns
for col in ['rotator_angle', 'alt', 'alt_deg', 'az'] + thermal_vars:
    n_valid = df[col].notna().sum() if col in df.columns else 0
    flag = 'OK' if col in df.columns else 'MISSING'
    print(f'  {col:<25s} {flag:<8s} valid={n_valid}/{len(df)}')

<a id='thermal'></a>
## Thermal Correlations

Reproduce the DZ-vs-thermal scatter plots from
`intrinsics_thermal_correlations.ipynb` over the combined multi-chunk dataset.
Each page shows one DZ coefficient against every thermal variable, with a
linear fit line and Pearson correlation `r`.

In [ ]:
def thermal_scatter_page(df, dz_col, k, j, thermal_vars):
    """Make a page of scatter plots: dz_col vs every thermal variable."""
    have = [tv for tv in thermal_vars if tv in df.columns]
    n = len(have)
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2 * nrows))
    fig.suptitle(
        f'DZ (k={k}, j={j}) = {dz_col}  vs thermal variables',
        fontsize=13, y=1.01)
    axes = np.atleast_1d(axes).ravel()
    for idx, tv in enumerate(have):
        ax = axes[idx]
        m = df[dz_col].notna() & df[tv].notna()
        x = df.loc[m, tv].values
        y = df.loc[m, dz_col].values
        ax.scatter(x, y, s=10, alpha=0.6, edgecolors='none')
        if len(x) > 2:
            c = np.polyfit(x, y, 1)
            r = np.corrcoef(x, y)[0, 1]
            xf = np.linspace(x.min(), x.max(), 50)
            ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
            ax.text(0.05, 0.92, f'r = {r:.3f}', transform=ax.transAxes,
                    fontsize=9, va='top',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))
        ax.set_xlabel(tv, fontsize=8)
        ax.set_ylabel(f'(k={k},j={j}) [µm]', fontsize=8)
        ax.tick_params(labelsize=7)
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)
    return fig

In [ ]:
thermal_figs = []
for k, j in dz_terms_for_thermal:
    col = f'{dz_prefix}_z{j}_c{k}'
    if col not in df.columns:
        print(f'Skipping {col}: not in table')
        continue
    if not any(tv in df.columns for tv in thermal_vars):
        print('No thermal variables present in tables — skipping section.')
        break
    fig = thermal_scatter_page(df, col, k, j, thermal_vars)
    thermal_figs.append(fig)
    plt.show()

<a id='corr'></a>
## Full DZ Correlation Analysis

Compute the pairwise Pearson correlation matrix over all DZ coefficients,
display it as a heatmap, then extract the `top_n_corr` largest off-diagonal
correlations (by `|r|`) and scatter-plot each pair.

In [ ]:
corr_df = df[dz_cols].dropna()
print(f'Correlation sample size: {len(corr_df)} visits, {len(dz_cols)} DZ terms')
corr_matrix = corr_df.corr(method='pearson')

fig_corr, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
labels = [short_label(c, dz_prefix) for c in dz_cols]
ax.set_xticks(range(len(dz_cols)))
ax.set_xticklabels(labels, rotation=90, fontsize=6)
ax.set_yticks(range(len(dz_cols)))
ax.set_yticklabels(labels, fontsize=6)
ax.set_title(f'Pearson correlation — all {dz_prefix} DZ coefficients ({coord_sys})')
plt.colorbar(im, ax=ax, shrink=0.8, label='r')
plt.show()

In [ ]:
# Extract upper-triangular off-diagonal correlations
n = len(dz_cols)
iu, ju = np.triu_indices(n, k=1)
rs = corr_matrix.values[iu, ju]
order = np.argsort(np.abs(rs))[::-1]
top = order[:top_n_corr]

top_pairs = []
print(f'Top {top_n_corr} |r| pairs:')
print(f'{"#":>3} {"col_a":<22} {"col_b":<22} {"r":>7}')
for rank, idx in enumerate(top, 1):
    a = dz_cols[iu[idx]]
    b = dz_cols[ju[idx]]
    r = rs[idx]
    top_pairs.append((a, b, r))
    print(f'{rank:>3} {a:<22} {b:<22} {r:>7.3f}')

In [ ]:
# Scatter plots of the top pairs in a grid
ncols = 2
nrows = int(np.ceil(top_n_corr / ncols))
fig_top, axes = plt.subplots(nrows, ncols, figsize=(11, 4.2 * nrows))
axes = np.atleast_1d(axes).ravel()
for idx, (a, b, r) in enumerate(top_pairs):
    ax = axes[idx]
    m = df[a].notna() & df[b].notna()
    x = df.loc[m, a].values
    y = df.loc[m, b].values
    ax.scatter(x, y, s=12, alpha=0.6, edgecolors='none')
    c = np.polyfit(x, y, 1)
    xf = np.linspace(x.min(), x.max(), 50)
    ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
    ax.set_xlabel(f'{a}\n{short_label(a, dz_prefix)} [µm]', fontsize=9)
    ax.set_ylabel(f'{b}\n{short_label(b, dz_prefix)} [µm]', fontsize=9)
    ax.set_title(f'r = {r:+.3f}   (n={m.sum()})', fontsize=10)
for idx in range(len(top_pairs), len(axes)):
    axes[idx].set_visible(False)
fig_top.suptitle(
    f'Top {top_n_corr} DZ–DZ correlations ({dz_prefix}, {coord_sys})',
    fontsize=13, y=1.005)
plt.show()

### Expected Astigmatism-Symmetry Correlations

Scatter plots of DZ coefficient pairs that should be correlated by the
astigmatism symmetry between focal Noll k = 5 ↔ 6 and pupil Noll j = 5 ↔ 6
(`expected_astig_pairs`).

In [ ]:
ncols = 3
nrows = int(np.ceil(len(expected_astig_pairs) / ncols))
fig_expected, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4.0 * nrows))
axes = np.atleast_1d(axes).ravel()

for idx, ((ka, ja), (kb, jb)) in enumerate(expected_astig_pairs):
    a = f'{dz_prefix}_z{ja}_c{ka}'
    b = f'{dz_prefix}_z{jb}_c{kb}'
    ax = axes[idx]
    if a not in df.columns or b not in df.columns:
        ax.set_visible(False)
        print(f'Skipping ({ka},{ja}) vs ({kb},{jb}): column missing')
        continue
    m = df[a].notna() & df[b].notna()
    x = df.loc[m, a].values
    y = df.loc[m, b].values
    r = np.corrcoef(x, y)[0, 1] if len(x) > 2 else np.nan
    ax.scatter(x, y, s=12, alpha=0.6, edgecolors='none')
    if len(x) > 2:
        c = np.polyfit(x, y, 1)
        xf = np.linspace(x.min(), x.max(), 50)
        ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
    ax.axhline(0, color='gray', lw=0.5, alpha=0.5)
    ax.axvline(0, color='gray', lw=0.5, alpha=0.5)
    ax.set_xlabel(f'(k={ka}, j={ja})  [µm]', fontsize=10)
    ax.set_ylabel(f'(k={kb}, j={jb})  [µm]', fontsize=10)
    ax.set_title(f'r = {r:+.3f}   (n={m.sum()})', fontsize=11)

for idx in range(len(expected_astig_pairs), len(axes)):
    axes[idx].set_visible(False)

fig_expected.suptitle(
    f'Expected astigmatism-symmetry DZ correlations ({dz_prefix}, {coord_sys})',
    fontsize=13, y=1.005)
plt.show()

<a id='lut'></a>
## LUT: DZ vs Rotator and Elevation

Inputs to a future Look-Up Table for AOS feed-forward correction.

* **Rotator scan** — plot every DZ coefficient vs `rotator_angle` for visits
  with `alt` within ±3 deg of 70 deg.
* **Elevation scan** — plot every DZ coefficient vs `alt` for visits with
  `rotator_angle` within ±3 deg of 0 deg.

In [ ]:
def grid_scan_page(df, x_col, dz_cols, prefix, title, x_label, x_unit='deg'):
    """Plot every DZ coefficient vs x_col in a tight grid."""
    n = len(dz_cols)
    ncols = 6
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(2.3 * ncols, 1.9 * nrows),
                             sharex=True)
    axes = np.atleast_1d(axes).ravel()
    for idx, col in enumerate(dz_cols):
        ax = axes[idx]
        m = df[col].notna() & df[x_col].notna()
        x = df.loc[m, x_col].values
        y = df.loc[m, col].values
        ax.scatter(x, y, s=6, alpha=0.6, edgecolors='none')
        ax.axhline(0, color='gray', lw=0.5, alpha=0.6)
        ax.set_title(short_label(col, prefix) if prefix in col
                     else col, fontsize=7)
        ax.tick_params(labelsize=6)
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)
    fig.suptitle(title, fontsize=12, y=1.005)
    fig.supxlabel(f'{x_label} [{x_unit}]', fontsize=10)
    fig.supylabel('DZ coefficient [µm]', fontsize=10)
    return fig

In [ ]:
# DZ vs rotator angle, at fixed elevation band (all in degrees)
elev_lo = elev_center_for_rotator_scan - elev_tol
elev_hi = elev_center_for_rotator_scan + elev_tol
mask_elev = df['alt_deg'].between(elev_lo, elev_hi)
df_rot = df[mask_elev & df['rotator_angle'].notna()].copy()
print(f'Elevation slice {elev_lo:.0f}–{elev_hi:.0f} deg: '
      f'{len(df_rot)}/{len(df)} visits')

if len(df_rot) >= 5:
    fig_rot = grid_scan_page(
        df_rot, 'rotator_angle', dz_cols, dz_prefix,
        title=(f'DZ vs rotator angle  (elevation = '
               f'{elev_center_for_rotator_scan:.0f} ± {elev_tol:.0f} deg, '
               f'n={len(df_rot)}, {coord_sys})'),
        x_label='rotator angle')
    plt.show()
else:
    fig_rot = None
    print('Too few visits in elevation slice for plot.')

In [ ]:
# DZ vs elevation (degrees), at fixed rotator-angle band
rot_lo = rot_center_for_elev_scan - rot_tol
rot_hi = rot_center_for_elev_scan + rot_tol
mask_rot = df['rotator_angle'].between(rot_lo, rot_hi)
df_elev = df[mask_rot & df['alt_deg'].notna()].copy()
print(f'Rotator slice {rot_lo:.0f}–{rot_hi:.0f} deg: '
      f'{len(df_elev)}/{len(df)} visits')

if len(df_elev) >= 5:
    fig_elev = grid_scan_page(
        df_elev, 'alt_deg', dz_cols, dz_prefix,
        title=(f'DZ vs elevation  (rotator = '
               f'{rot_center_for_elev_scan:.0f} ± {rot_tol:.0f} deg, '
               f'n={len(df_elev)}, {coord_sys})'),
        x_label='elevation')
    plt.show()
else:
    fig_elev = None
    print('Too few visits in rotator slice for plot.')

<a id='radial'></a>
## Cylindrical (Radial) Pupil Zernike Combinations

The Noll-indexed Zernikes pair azimuthal cos/sin terms. The amplitude
$Z_m^{(r)} = \sqrt{Z_\text{cos}^2 + Z_\text{sin}^2}$ is rotationally invariant
and carries the image-quality information; the associated angle
$\arctan(Z_\text{sin}/Z_\text{cos})$ is less interesting for AOS diagnostics.

Purely radial terms (m = 0) are left unchanged. Combining only the **pupil**
Zernikes (keeping focal-plane Zernikes separate, since any camera-side optical
effect may add rotator-angle dependence beyond the OCS→CCS transform), the 21
pupil Noll indices Z4–Z24 reduce to **12 radial terms**:

| Short name        | Description           | Noll indices combined |
|-------------------|-----------------------|-----------------------|
| `Z4`              | defocus               | 4                     |
| `Z_astig`         | astigmatism           | 5, 6                  |
| `Z_coma`          | coma                  | 7, 8                  |
| `Z_trefoil`       | trefoil               | 9, 10                 |
| `Z11`             | spherical             | 11                    |
| `Z_2ndastig`      | 2nd astigmatism       | 12, 13                |
| `Z_tetrafoil`     | tetrafoil             | 14, 15                |
| `Z_2ndcoma`       | 2nd coma              | 16, 17                |
| `Z_2ndtrefoil`    | 2nd trefoil           | 18, 19                |
| `Z_pentafoil`     | pentafoil             | 20, 21                |
| `Z22`             | 2nd spherical         | 22                    |
| `Z_3rdastig`      | 3rd astigmatism       | 23, 24                |

For a DZ coefficient the combination is applied per focal-plane Noll index `k`:
$c_k^{(\text{radial})} = \sqrt{c_k^{(\text{cos})2} + c_k^{(\text{sin})2}}$.

In [ ]:
# Radial pupil Zernike definitions (Noll convention)
# Each entry: (short_name, [list of Noll indices to combine])
RADIAL_PUPIL_DEFS = [
    ('Z4',            [4]),
    ('Z_astig',       [5, 6]),
    ('Z_coma',        [7, 8]),
    ('Z_trefoil',     [9, 10]),
    ('Z11',           [11]),
    ('Z_2ndastig',    [12, 13]),
    ('Z_tetrafoil',   [14, 15]),
    ('Z_2ndcoma',     [16, 17]),
    ('Z_2ndtrefoil',  [18, 19]),
    ('Z_pentafoil',   [20, 21]),
    ('Z22',           [22]),
    ('Z_3rdastig',    [23, 24]),
]


def radial_combine(values_list):
    """Return sqrt(sum(v**2)) with NaN propagation element-wise."""
    arr = np.asarray(values_list, dtype=float)
    if arr.ndim == 1:
        return float(np.sqrt(np.nansum(arr**2))) if not np.any(np.isnan(arr)) else np.nan
    # axis=0: combine across the Noll-index axis
    nan_mask = np.isnan(arr).any(axis=0)
    out = np.sqrt(np.nansum(arr**2, axis=0))
    out[nan_mask] = np.nan
    return out


def add_radial_dz_columns(df, prefix, radial_defs, max_k=6):
    """Add radial-combined DZ columns to df.

    Column name: '{prefix}_{short_name}_c{k}' for k = 1..max_k.
    Only adds a column if ALL required source columns are present.

    Returns list of newly-added column names.
    """
    added = []
    for short_name, nolls in radial_defs:
        for k in range(1, max_k + 1):
            src_cols = [f'{prefix}_z{j}_c{k}' for j in nolls]
            missing = [c for c in src_cols if c not in df.columns]
            if missing:
                continue
            new_col = f'{prefix}_{short_name}_c{k}'
            vals = np.stack([df[c].values.astype(float) for c in src_cols])
            df[new_col] = radial_combine(vals)
            added.append(new_col)
    return added


# Determine max focal k from prefix
max_focal_k = 6 if dz_prefix == 'z1toz6' else 3
radial_cols = add_radial_dz_columns(df, dz_prefix, RADIAL_PUPIL_DEFS,
                                    max_k=max_focal_k)
print(f'Added {len(radial_cols)} radial DZ columns '
      f'({len(RADIAL_PUPIL_DEFS)} pupil terms × {max_focal_k} focal k)')
print('Sample:', radial_cols[:6], '...')

### Correlation Analysis of Radial DZ Terms

Repeat the Pearson-correlation study (full heatmap + top-N scatter plots) using
the 12 radial pupil combinations with all focal-plane Noll indices retained.

In [ ]:
def radial_label(col, prefix):
    """Label like 'Z_astig (k=3)' from 'z1toz6_Z_astig_c3'."""
    m = re.match(rf'^{re.escape(prefix)}_(.+)_c(\d+)$', col)
    if not m:
        return col
    return f'{m.group(1)} (k={m.group(2)})'


radial_corr_df = df[radial_cols].dropna()
print(f'Radial correlation sample size: {len(radial_corr_df)} visits, '
      f'{len(radial_cols)} radial DZ terms')
radial_corr_matrix = radial_corr_df.corr(method='pearson')

fig_rcorr, ax = plt.subplots(figsize=(12, 11))
im = ax.imshow(radial_corr_matrix.values, cmap='RdBu_r',
               vmin=-1, vmax=1, aspect='auto')
rlabels = [radial_label(c, dz_prefix) for c in radial_cols]
ax.set_xticks(range(len(radial_cols)))
ax.set_xticklabels(rlabels, rotation=90, fontsize=6)
ax.set_yticks(range(len(radial_cols)))
ax.set_yticklabels(rlabels, fontsize=6)
ax.set_title(
    f'Pearson correlation — radial-combined pupil DZ terms ({coord_sys})')
plt.colorbar(im, ax=ax, shrink=0.8, label='r')
plt.show()

In [ ]:
# Top-N radial-DZ correlations
nr = len(radial_cols)
iu_r, ju_r = np.triu_indices(nr, k=1)
rs_r = radial_corr_matrix.values[iu_r, ju_r]
order_r = np.argsort(np.abs(rs_r))[::-1]
top_r = order_r[:top_n_corr]

top_pairs_r = []
print(f'Top {top_n_corr} |r| pairs (radial terms):')
print(f'{"#":>3} {"col_a":<30} {"col_b":<30} {"r":>7}')
for rank, idx in enumerate(top_r, 1):
    a = radial_cols[iu_r[idx]]
    b = radial_cols[ju_r[idx]]
    r = rs_r[idx]
    top_pairs_r.append((a, b, r))
    print(f'{rank:>3} {a:<30} {b:<30} {r:>7.3f}')

# Scatter grid
ncols = 2
nrows = int(np.ceil(top_n_corr / ncols))
fig_rtop, axes = plt.subplots(nrows, ncols, figsize=(11, 4.2 * nrows))
axes = np.atleast_1d(axes).ravel()
for idx, (a, b, r) in enumerate(top_pairs_r):
    ax = axes[idx]
    m = df[a].notna() & df[b].notna()
    x = df.loc[m, a].values
    y = df.loc[m, b].values
    ax.scatter(x, y, s=12, alpha=0.6, edgecolors='none')
    c = np.polyfit(x, y, 1)
    xf = np.linspace(x.min(), x.max(), 50)
    ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
    ax.set_xlabel(f'{radial_label(a, dz_prefix)} [µm]', fontsize=9)
    ax.set_ylabel(f'{radial_label(b, dz_prefix)} [µm]', fontsize=9)
    ax.set_title(f'r = {r:+.3f}   (n={m.sum()})', fontsize=10)
for idx in range(len(top_pairs_r), len(axes)):
    axes[idx].set_visible(False)
fig_rtop.suptitle(
    f'Top {top_n_corr} radial DZ–DZ correlations ({coord_sys})',
    fontsize=13, y=1.005)
plt.show()

### Radial-Combination Trio Plots

For each radial pupil term, construct focal-plane Trio maps:

* **Data** — measured per-donut zk minus the fitted linear (focal-plane k=1,2,3)
  component, combined radially across the Noll pair.
* **Intrinsic** — the batoid intrinsic model value, radially combined.
* **Data – Intrinsic** — residual.

Reads the HDF5 `donuts` tables for every chunk and (re)constructs the
per-donut fitted zk from the matching `_fits.parquet` (subtracting only
focal-plane Noll k = 1, 2, 3 — the `z1toz3` fit).

In [ ]:
from astropy.table import QTable, vstack
from scipy.stats import binned_statistic_2d

# Pull helpers from our code/ package
sys.path.insert(0, 'code')
from dz_plotting import reconstruct_zk_fit
from dz_fitting import derive_noll_indices


def hdf5_for_fitfile(fit_path, coord_sys):
    """Derive the HDF5 path corresponding to a fit parquet."""
    p = Path(fit_path)
    stem = p.stem
    if coord_sys == 'CCS' and stem.endswith('_ccs_fits'):
        base = stem[:-len('_ccs_fits')]
    elif stem.endswith('_fits'):
        base = stem[:-len('_fits')]
    else:
        base = stem
    return p.parent / f'{base}.hdf5'


def load_donuts_for_trio(fit_files, coord_sys, fit_prefix='z1toz3'):
    """Load 'donuts' tables for all chunks and attach reconstructed zk_fit.

    Returns (aosTable, iZs, iZidx) where iZs is the Noll-index list.
    """
    tables = []
    iZs_ref = None
    for fit_path in fit_files:
        h5 = hdf5_for_fitfile(fit_path, coord_sys)
        if not h5.exists():
            print(f'  [skip] {h5.name} not found')
            continue
        aos = QTable.read(str(h5), path='donuts')
        visit_info = QTable.read(str(h5), path='visits')

        # Noll indices
        nZk = np.stack(aos[f'zk_{coord_sys}']).shape[1]
        noll_arr = (np.array(visit_info['nollIndices'][0])
                    if 'nollIndices' in visit_info.colnames else None)
        iZs, iZidx = derive_noll_indices(nZk, noll_arr)
        if iZs_ref is None:
            iZs_ref, iZidx_ref = iZs, iZidx
        elif iZs != iZs_ref:
            print(f'  [warn] {h5.name}: iZs {iZs} != {iZs_ref}; skipping')
            continue

        # Fit table and reconstruct zk_fit_{fit_prefix}
        ft = QTable.read(str(fit_path), format='parquet')
        max_k = 3 if fit_prefix == 'z1toz3' else 6
        reconstruct_zk_fit(aos, ft, coord_sys, iZs,
                           prefix=fit_prefix, max_focal_noll=max_k)
        tables.append(aos)
        print(f'  {h5.name}: {len(aos)} donuts')

    if not tables:
        raise FileNotFoundError('No HDF5 donut tables found')
    combined = vstack(tables, metadata_conflicts='silent')
    print(f'\nCombined donuts: {len(combined)} rows')
    return combined, iZs_ref, iZidx_ref

In [ ]:
# Load donut-level data for all chunks.  Use the z1toz3 fit so that only
# focal-plane k = 1, 2, 3 components are subtracted from the data.
aosTable_all, iZs_all, iZidx_all = load_donuts_for_trio(
    fit_files, coord_sys, fit_prefix='z1toz3')

In [ ]:
def plot_radial_trio(aosTable, short_name, nolls, iZidx, coord_sys,
                     fit_prefix='z1toz3', statistic='median',
                     plo=4.0, phi=96.0):
    """Trio plot for a radial-combined pupil Zernike term.

    Each of the three panels bins per-donut values on the focal plane:
      - Data: sqrt(sum((zk - zk_fit)^2)) over Noll indices in `nolls`.
      - Intrinsic: sqrt(sum(zk_intrinsic^2)).
      - Data - Intrinsic: signed difference (per-donut, then combined).
    """
    nsteps = 18 * 4 + 1
    fpradius = 1.8
    xbins = np.linspace(-fpradius, fpradius, nsteps)
    ybins = np.linspace(-fpradius, fpradius, nsteps)

    xval = np.rad2deg(np.array(aosTable[f'thy_{coord_sys}_extra']))
    yval = np.rad2deg(np.array(aosTable[f'thx_{coord_sys}_extra']))

    zk_data_all = np.stack(aosTable[f'zk_{coord_sys}'])
    zk_fit_all = np.stack(aosTable[f'zk_fit_{fit_prefix}'])
    zk_model_all = np.stack(aosTable[f'zk_intrinsic_{coord_sys}'])

    # Combine across Noll indices in quadrature (per donut)
    data_stack = []
    model_stack = []
    for j in nolls:
        if j not in iZidx:
            return None  # skip if Noll index missing
        idx = iZidx[j]
        data_stack.append(zk_data_all[:, idx] - zk_fit_all[:, idx])
        model_stack.append(zk_model_all[:, idx])
    data_stack = np.stack(data_stack)   # (nNoll, nDonut)
    model_stack = np.stack(model_stack)

    zval_data = np.sqrt(np.nansum(data_stack ** 2, axis=0))
    zval_model = np.sqrt(np.nansum(model_stack ** 2, axis=0))
    zval_residual = zval_data - zval_model

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    for ax, zval, cmap, title_str in [
        (axes[0], zval_data, 'viridis',
         f'{short_name} Data (linear fit subtracted)'),
        (axes[1], zval_model, 'viridis', f'{short_name} Intrinsic Model'),
        (axes[2], zval_residual, 'RdBu_r', f'{short_name} Data − Model'),
    ]:
        stat_val, _, _, _ = binned_statistic_2d(
            xval, yval, zval, statistic=statistic, bins=[xbins, ybins])
        vmin, vmax = np.nanpercentile(zval, [plo, phi])
        im = ax.imshow(stat_val.T, origin='lower',
                       extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
                       cmap=cmap, interpolation='none', aspect='equal',
                       vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.8, label='µm')
        ax.set_xlabel(f'thy_{coord_sys} [deg]')
        ax.set_ylabel(f'thx_{coord_sys} [deg]')
        ax.set_title(title_str)
        ax.set_aspect('equal')

    noll_str = ','.join(str(j) for j in nolls)
    fig.suptitle(
        f'Radial {short_name}  (Noll {noll_str}, {coord_sys}, {statistic})',
        fontsize=14, y=1.02)
    print(f'{short_name}: Data σ={np.nanstd(zval_data):.3f}, '
          f'Model σ={np.nanstd(zval_model):.3f}, '
          f'Resid σ={np.nanstd(zval_residual):.3f} µm')
    return fig

In [ ]:
trio_figs = []
for short_name, nolls in RADIAL_PUPIL_DEFS:
    if not all(j in iZidx_all for j in nolls):
        print(f'{short_name}: skip (Noll {nolls} not all present)')
        continue
    fig = plot_radial_trio(aosTable_all, short_name, nolls,
                           iZidx_all, coord_sys,
                           fit_prefix='z1toz3', statistic='median')
    if fig is not None:
        trio_figs.append(fig)
        plt.show()

<a id='output'></a>
## Output

Combine all figures into a single PDF.

In [ ]:
Path(output_pdf).parent.mkdir(parents=True, exist_ok=True)

with PdfPages(output_pdf) as pdf:
    for fig in thermal_figs:
        pdf.savefig(fig, bbox_inches='tight')
    pdf.savefig(fig_corr, bbox_inches='tight')
    pdf.savefig(fig_top, bbox_inches='tight')
    pdf.savefig(fig_expected, bbox_inches='tight')
    if fig_rot is not None:
        pdf.savefig(fig_rot, bbox_inches='tight')
    if fig_elev is not None:
        pdf.savefig(fig_elev, bbox_inches='tight')
    pdf.savefig(fig_rcorr, bbox_inches='tight')
    pdf.savefig(fig_rtop, bbox_inches='tight')
    for fig in trio_figs:
        pdf.savefig(fig, bbox_inches='tight')

print(f'Wrote PDF: {output_pdf}')